### Objective: To analyze how market sentiment (Fear/Greed) relates to trader behavior and performance on Hyperliquid. The goal is to uncover patterns that could inform smarter trading strategies.

In [73]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = r"C:\Users\HP\Desktop\internship_submission\data"

## Part A — Data Preparation

In [74]:
# Load sentiment data
sentiment = pd.read_csv(f"{DATA_DIR}/fear_greed_index.csv")
print(f"\n--- Sentiment Dataset ---")
print(f"Shape: {sentiment.shape}")
print(f"Columns: {list(sentiment.columns)}")
print(f"Dtypes:\n{sentiment.dtypes}")
print(f"\nMissing values:\n{sentiment.isnull().sum()}")
print(f"Duplicates: {sentiment.duplicated().sum()}")
print(f"\nFirst 5 rows:\n{sentiment.head()}")
print(f"\nDate range: {sentiment['date'].min()} to {sentiment['date'].max()}")
print(f"\nClassification distribution:\n{sentiment['classification'].value_counts()}")
print(f"\nValue stats:\n{sentiment['value'].describe()}")


--- Sentiment Dataset ---
Shape: (2644, 4)
Columns: ['timestamp', 'value', 'classification', 'date']
Dtypes:
timestamp          int64
value              int64
classification    object
date              object
dtype: object

Missing values:
timestamp         0
value             0
classification    0
date              0
dtype: int64
Duplicates: 0

First 5 rows:
    timestamp  value classification        date
0  1517463000     30           Fear  2018-02-01
1  1517549400     15   Extreme Fear  2018-02-02
2  1517635800     40           Fear  2018-02-03
3  1517722200     24   Extreme Fear  2018-02-04
4  1517808600     11   Extreme Fear  2018-02-05

Date range: 2018-02-01 to 2025-05-02

Classification distribution:
classification
Fear             781
Greed            633
Extreme Fear     508
Neutral          396
Extreme Greed    326
Name: count, dtype: int64

Value stats:
count    2644.000000
mean       46.981089
std        21.827680
min         5.000000
25%        28.000000
50%        46.00

In [75]:
# Load trader data
trades = pd.read_csv(f"{DATA_DIR}/historical_data.csv")
print(f"\n\n--- Trader Dataset ---")
print(f"Shape: {trades.shape}")
print(f"Columns: {list(trades.columns)}")
print(f"Dtypes:\n{trades.dtypes}")
print(f"\nMissing values:\n{trades.isnull().sum()}")
print(f"Duplicates: {trades.duplicated().sum()}")
print(f"\nFirst 3 rows:\n{trades.head(3).to_string()}")



--- Trader Dataset ---
Shape: (211224, 16)
Columns: ['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL', 'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID', 'Timestamp']
Dtypes:
Account              object
Coin                 object
Execution Price     float64
Size Tokens         float64
Size USD            float64
Side                 object
Timestamp IST        object
Start Position      float64
Direction            object
Closed PnL          float64
Transaction Hash     object
Order ID              int64
Crossed                bool
Fee                 float64
Trade ID            float64
Timestamp           float64
dtype: object

Missing values:
Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order 

In [76]:
# Parse dates
sentiment['date'] = pd.to_datetime(sentiment['date'])

In [77]:
# Parse trader timestamps - format is DD-MM-YYYY HH:MM
trades['Timestamp IST'] = pd.to_datetime(trades['Timestamp IST'], format='%d-%m-%Y %H:%M', errors='coerce')
trades['date'] = trades['Timestamp IST'].dt.date
trades['date'] = pd.to_datetime(trades['date'])

In [78]:
print(f"\nTrader date range: {trades['date'].min()} to {trades['date'].max()}")
print(f"Unique accounts: {trades['Account'].nunique()}")
print(f"Unique coins: {trades['Coin'].nunique()}")
print(f"Coins traded: {trades['Coin'].unique()}")
print(f"\nSide distribution:\n{trades['Side'].value_counts()}")
print(f"\nDirection distribution:\n{trades['Direction'].value_counts()}")


Trader date range: 2023-05-01 00:00:00 to 2025-05-01 00:00:00
Unique accounts: 32
Unique coins: 246
Coins traded: ['@107' 'AAVE' 'DYDX' 'AIXBT' 'GMX' 'EIGEN' 'HYPE' 'SOL' 'SUI' 'DOGE'
 'ETH' 'kPEPE' 'TRUMP' 'ONDO' 'ENA' 'LINK' 'XRP' 'S' 'BNB' 'BERA' 'WIF'
 'LAYER' 'MKR' 'KAITO' 'IP' 'JUP' 'USUAL' 'ADA' 'BTC' 'PURR/USDC' 'ZRO'
 '@7' '@19' '@21' '@44' '@48' '@11' '@15' '@46' '@61' '@28' '@45' '@9'
 '@41' '@38' 'kSHIB' 'GRASS' 'TAO' 'AVAX' '@2' '@6' '@8' '@10' '@12' '@16'
 '@17' '@35' '@26' '@24' '@32' '@29' '@31' '@33' '@34' '@36' '@37' '@47'
 '@53' '@74' 'RUNE' 'CANTO' 'NTRN' 'BLUR' 'ZETA' 'MINA' 'MANTA' 'RNDR'
 'WLD' 'kBONK' 'ALT' 'INJ' 'STG' 'ZEN' 'MAVIA' 'PIXEL' 'ILV' 'FET' 'STRK'
 'CAKE' 'STX' 'ACE' 'PENDLE' 'AR' 'XAI' 'APE' 'MEME' 'NEAR' 'SEI' 'FTM'
 'MYRO' 'BIGTIME' 'IMX' 'BADGER' 'POLYX' 'OP' 'TNSR' 'MAV' 'TIA' 'MERL'
 'TON' 'PURR' 'ME' 'CRV' 'BRETT' 'CHILLGUY' 'MOODENG' 'VIRTUAL' 'COMP'
 'FARTCOIN' 'AI16Z' 'GRIFFAIN' 'ZEREBRO' 'SPX' 'MELANIA' 'PENGU' 'JELLY'
 'VVV' 'VINE' 'TST'

In [79]:
# Merge datasets
merged = trades.merge(sentiment[['date', 'value', 'classification']], on='date', how='inner')
print(f"\n\n--- Merged Dataset ---")
print(f"Shape after merge: {merged.shape}")
print(f"Date range in merged: {merged['date'].min()} to {merged['date'].max()}")
print(f"Unique dates in merged: {merged['date'].nunique()}")
print(f"\nSentiment distribution in merged data:\n{merged['classification'].value_counts()}")




--- Merged Dataset ---
Shape after merge: (211218, 19)
Date range in merged: 2023-05-01 00:00:00 to 2025-05-01 00:00:00
Unique dates in merged: 479

Sentiment distribution in merged data:
classification
Fear             61837
Greed            50303
Extreme Greed    39992
Neutral          37686
Extreme Fear     21400
Name: count, dtype: int64


In [80]:
print("\n\n" + "=" * 70)
print("KEY METRICS")
print("=" * 70)

# Simplify sentiment to Fear vs Greed
def simplify_sentiment(cls):
    if cls in ['Extreme Fear', 'Fear']:
        return 'Fear'
    elif cls in ['Extreme Greed', 'Greed']:
        return 'Greed'
    else:
        return 'Neutral'
merged['sentiment_group'] = merged['classification'].apply(simplify_sentiment)
print(f"\nSimplified sentiment in merged:\n{merged['sentiment_group'].value_counts()}")




KEY METRICS

Simplified sentiment in merged:
sentiment_group
Greed      90295
Fear       83237
Neutral    37686
Name: count, dtype: int64


In [81]:
# Convert Closed PnL to numeric
merged['Closed PnL'] = pd.to_numeric(merged['Closed PnL'], errors='coerce')
merged['Size USD'] = pd.to_numeric(merged['Size USD'], errors='coerce')
merged['Fee'] = pd.to_numeric(merged['Fee'], errors='coerce')

In [82]:
# Net PnL = Closed PnL - Fee
merged['Net PnL'] = merged['Closed PnL'] - merged['Fee']

In [83]:
# Daily PnL per account
daily_pnl = merged.groupby(['date', 'Account', 'sentiment_group', 'classification']).agg(
    total_pnl=('Closed PnL', 'sum'),
    total_fee=('Fee', 'sum'),
    net_pnl=('Net PnL', 'sum'),
    num_trades=('Closed PnL', 'count'),
    avg_trade_size=('Size USD', 'mean'),
    total_volume=('Size USD', 'sum'),
    wins=('Closed PnL', lambda x: (x > 0).sum()),
    losses=('Closed PnL', lambda x: (x < 0).sum()),
    zero_pnl=('Closed PnL', lambda x: (x == 0).sum()),
).reset_index()

daily_pnl['win_rate'] = daily_pnl['wins'] / (daily_pnl['wins'] + daily_pnl['losses']).replace(0, np.nan)

In [84]:
# Trades with actual PnL (non-zero = closed trades)
closed_trades = merged[merged['Closed PnL'] != 0]
open_trades = merged[merged['Closed PnL'] == 0]

print(f"\nTotal trades: {len(merged)}")
print(f"Trades with Closed PnL != 0: {len(closed_trades)}")
print(f"Trades with Closed PnL == 0 (open/new positions): {len(open_trades)}")


Total trades: 211218
Trades with Closed PnL != 0: 104402
Trades with Closed PnL == 0 (open/new positions): 106816


Sentiment Dataset has 2644 rows and 4 columns.
Trader Dataset has 211224 rows and 16 columns.
No missing values in either dataset. No duplicates.
32 unique accounts trading 246 different coins
104,402 trades with realized PnL (Closed PnL ≠ 0), 106,816 are position-opening trades (PnL = 0)

## Part B — Analysis


In [85]:
# Q1: Performance on Fear vs Greed days
print("\n--- Q1: Performance (PnL, Win Rate) by Sentiment ---")

perf_by_sentiment = merged.groupby('sentiment_group').agg(
    total_pnl=('Closed PnL', 'sum'),
    mean_pnl=('Closed PnL', 'mean'),
    median_pnl=('Closed PnL', 'median'),
    total_trades=('Closed PnL', 'count'),
    positive_pnl_trades=('Closed PnL', lambda x: (x > 0).sum()),
    negative_pnl_trades=('Closed PnL', lambda x: (x < 0).sum()),
    zero_pnl_trades=('Closed PnL', lambda x: (x == 0).sum()),
    total_volume=('Size USD', 'sum'),
    avg_size=('Size USD', 'mean'),
).reset_index()

# Win rate only among trades that have non-zero PnL
perf_by_sentiment['win_rate'] = perf_by_sentiment['positive_pnl_trades'] / (perf_by_sentiment['positive_pnl_trades'] + perf_by_sentiment['negative_pnl_trades']).replace(0, np.nan)

print(perf_by_sentiment.to_string(index=False))



--- Q1: Performance (PnL, Win Rate) by Sentiment ---
sentiment_group    total_pnl  mean_pnl  median_pnl  total_trades  positive_pnl_trades  negative_pnl_trades  zero_pnl_trades  total_volume    avg_size  win_rate
           Fear 4.096266e+06 49.212077         0.0         83237                33950                 6264            43023  597809051.23 7182.011019  0.844233
          Greed 4.865301e+06 53.882281         0.0         90295                37952                 8077            44266  413047659.29 4574.424490  0.824524
        Neutral 1.292921e+06 34.307718         0.0         37686                14961                 3198            19527  180242063.08 4782.732661  0.823889


In [86]:
# By detailed classification
print("\n--- By Detailed Classification ---")
perf_detail = merged.groupby('classification').agg(
    total_pnl=('Closed PnL', 'sum'),
    mean_pnl=('Closed PnL', 'mean'),
    total_trades=('Closed PnL', 'count'),
    positive_trades=('Closed PnL', lambda x: (x > 0).sum()),
    negative_trades=('Closed PnL', lambda x: (x < 0).sum()),
    avg_size=('Size USD', 'mean'),
).reset_index()
perf_detail['win_rate'] = perf_detail['positive_trades'] / (perf_detail['positive_trades'] + perf_detail['negative_trades']).replace(0, np.nan)
print(perf_detail.to_string(index=False))


--- By Detailed Classification ---
classification    total_pnl  mean_pnl  total_trades  positive_trades  negative_trades    avg_size  win_rate
  Extreme Fear 7.391102e+05 34.537862         21400             7931             2475 5349.731843  0.762156
 Extreme Greed 2.715171e+06 67.892861         39992            18594             2259 3112.251565  0.891670
          Fear 3.357155e+06 54.290400         61837            26019             3789 7816.109931  0.872886
         Greed 2.150129e+06 42.743559         50303            19358             5818 5736.884375  0.768907
       Neutral 1.292921e+06 34.307718         37686            14961             3198 4782.732661  0.823889


In [87]:
# Drawdown proxy - max single-day loss by sentiment
print("\n--- Max Single Trade Loss by Sentiment ---")
max_loss = merged[merged['Closed PnL'] < 0].groupby('sentiment_group')['Closed PnL'].agg(['min', 'mean', 'count'])
print(max_loss)


--- Max Single Trade Loss by Sentiment ---
                          min        mean  count
sentiment_group                                 
Fear             -35681.74723 -196.346650   6264
Greed           -117990.10410 -164.613823   8077
Neutral          -24500.00000 -121.727849   3198


### Q1: Does performance differ between Fear vs Greed days?

Yes — the relationship is nuanced and reveals non-intuitive patterns.

### Key Findings

- Total PnL is higher on Greed days ($4.87M vs $4.10M)
- Win rate is higher on Fear days (84.4% vs 82.5%)
- Traders are 2.7x more active during Fear
- Extreme Greed shows highest efficiency (89.2% win rate)

In [88]:
#Q2: Behavior changes
print("\n\n--- Q2: Behavioral Changes by Sentiment ---")

# Trade frequency per day by sentiment
daily_counts = merged.groupby(['date', 'sentiment_group']).size().reset_index(name='trades_per_day')
freq_by_sent = daily_counts.groupby('sentiment_group')['trades_per_day'].agg(['mean', 'median', 'sum']).reset_index()
freq_by_sent.columns = ['sentiment_group', 'avg_trades_per_day', 'median_trades_per_day', 'total_trades']
print("\nTrade Frequency:")
print(freq_by_sent.to_string(index=False))



--- Q2: Behavioral Changes by Sentiment ---

Trade Frequency:
sentiment_group  avg_trades_per_day  median_trades_per_day  total_trades
           Fear          792.733333                  192.0         83237
          Greed          294.120521                   63.0         90295
        Neutral          562.477612                   50.0         37686


In [89]:
# Long/Short ratio
print("\nLong/Short distribution by Sentiment:")
side_sent = merged.groupby(['sentiment_group', 'Side']).size().unstack(fill_value=0)
print(side_sent)
print("\nLong/Short percentages:")
for sent in side_sent.index:
    row = side_sent.loc[sent]
    total = row.sum()
    for col in side_sent.columns:
        pct = row[col] / total * 100
        print(f"  {sent} - {col}: {row[col]} ({pct:.1f}%)")


Long/Short distribution by Sentiment:
Side               BUY   SELL
sentiment_group              
Fear             41205  42032
Greed            42516  47779
Neutral          18969  18717

Long/Short percentages:
  Fear - BUY: 41205 (49.5%)
  Fear - SELL: 42032 (50.5%)
  Greed - BUY: 42516 (47.1%)
  Greed - SELL: 47779 (52.9%)
  Neutral - BUY: 18969 (50.3%)
  Neutral - SELL: 18717 (49.7%)


In [90]:
# Direction breakdown
print("\nDirection distribution by Sentiment:")
dir_sent = merged.groupby(['sentiment_group', 'Direction']).size().unstack(fill_value=0)
print(dir_sent)


Direction distribution by Sentiment:
Direction        Auto-Deleveraging   Buy  Close Long  Close Short  \
sentiment_group                                                     
Fear                             0  4014       23501        12338   
Greed                            8  9817       15184        17819   
Neutral                          0  2885        9993         5850   

Direction        Liquidated Isolated Short  Long > Short  Open Long  \
sentiment_group                                                       
Fear                                     0            19      24829   
Greed                                    1            23      14844   
Neutral                                  0            15      10222   

Direction        Open Short   Sell  Settlement  Short > Long  \
sentiment_group                                                
Fear                  14061   4406           0            24   
Greed                 19327  13150           1            34   
Neut

In [91]:
# Position sizes by sentiment
print("\nAverage Trade Size (USD) by Sentiment:")
size_sent = merged.groupby('sentiment_group')['Size USD'].agg(['mean', 'median', 'std', 'max']).reset_index()
print(size_sent.to_string(index=False))


Average Trade Size (USD) by Sentiment:
sentiment_group        mean  median          std        max
           Fear 7182.011019 749.400 46166.174380 3921430.72
          Greed 4574.424490 552.200 23984.715663 2227114.71
        Neutral 4782.732661 547.655 37461.883466 3641180.84


### Q2: Do traders change behavior based on sentiment (trade frequency, leverage, long/short bias, position sizes)?

Ans: Yes, they do and in the following ways:

Traders are 2.7x MORE active on Fear days (793 trades/day vs 294 on Greed). This is OPPOSITE to the common assumption that traders retreat during fear. These Hyperliquid traders appear to be sophisticated — they increase activity during fear to capitalize on volatility.

The buy/sell ratio is relatively balanced across sentiment. On Greed days, there's a slight sell bias (52.9%), suggesting traders take profits during greed rather than opening new positions.

Average position size is 57% LARGER on Fear days ($7,182 vs $4,574). Combined with higher trade frequency, this means traders are deploying significantly more capital during fear periods. This is a contrarian, professional-level behavior pattern.

Short positions are significantly more common during Fear (18,373 shorts opened vs 13,254 during Greed — 38% more). Traders actively hedge or bet on downside during fearful periods.


In [102]:
# Q3: Trader Segments
print("\n\n--- Q3: Trader Segments ---")

# Per-account stats
account_stats = merged.groupby('Account').agg(
    total_trades=('Closed PnL', 'count'),
    total_pnl=('Closed PnL', 'sum'),
    mean_pnl=('Closed PnL', 'mean'),
    positive_trades=('Closed PnL', lambda x: (x > 0).sum()),
    negative_trades=('Closed PnL', lambda x: (x < 0).sum()),
    zero_trades=('Closed PnL', lambda x: (x == 0).sum()),
    avg_size=('Size USD', 'mean'),
    total_volume=('Size USD', 'sum'),
    unique_days=('date', 'nunique'),
    unique_coins=('Coin', 'nunique'),
    coins=('Coin', lambda x: list(x.unique())),
).reset_index()



--- Q3: Trader Segments ---


In [94]:
account_stats['win_rate'] = account_stats['positive_trades'] / (account_stats['positive_trades'] + account_stats['negative_trades']).replace(0, np.nan)
account_stats['trades_per_day'] = account_stats['total_trades'] / account_stats['unique_days']
account_stats['net_pnl'] = account_stats['total_pnl']

print(f"\nNumber of unique accounts: {len(account_stats)}")
print("\nTop 20 accounts by total volume:")
top_accounts = account_stats.sort_values('total_volume', ascending=False).head(20)
print(top_accounts[['Account', 'total_trades', 'total_pnl', 'win_rate', 'avg_size', 'total_volume', 'unique_days', 'unique_coins', 'trades_per_day']].to_string(index=False))


Number of unique accounts: 32

Top 20 accounts by total volume:
                                   Account  total_trades     total_pnl  win_rate     avg_size  total_volume  unique_days  unique_coins  trades_per_day
0x513b8629fe877bb581bf244e326a047b249c4ff1         12236  8.404226e+05  0.895476 34396.580284  420876556.36           39             4      313.743590
0x4f93fead39b70a1824f981a54d4e55b278e9f760          7584  3.089759e+05  0.926755 17098.171055  129672529.28          321            24       23.626168
0xb899e522b5715391ae1d4f137653e7906c5e2115          4838  2.248850e+04  0.803484 22504.555829  108877041.10           35             7      138.228571
0xbee1707d6b44d4d52bfe19e41f8a828645437aab         40184  8.360806e+05  0.763070  1844.211886   74107810.44          131            16      306.748092
0xbaaaf6571ab7d571043ff1e313a9609a10637864         21192  9.401638e+05  0.991197  3210.472831   68036340.24           28             3      756.857143
0x083384f897ee0f19899168e3b1b

In [95]:
# Segment: High vs Low volume traders
median_vol = account_stats['total_volume'].median()
account_stats['volume_segment'] = account_stats['total_volume'].apply(lambda x: 'High Volume' if x > median_vol else 'Low Volume')

In [96]:
print(f"\n\nVolume segments (median={median_vol:.0f}):")
seg_stats = account_stats.groupby('volume_segment').agg(
    count=('Account', 'count'),
    avg_pnl=('total_pnl', 'mean'),
    avg_win_rate=('win_rate', 'mean'),
    avg_trades=('total_trades', 'mean'),
).reset_index()
print(seg_stats.to_string(index=False))



Volume segments (median=11736837):
volume_segment  count       avg_pnl  avg_win_rate  avg_trades
   High Volume     16 511046.550787      0.823838  10541.6875
    Low Volume     16 129858.883544      0.876249   2659.4375


In [97]:
# Segment: Frequent vs Infrequent
median_freq = account_stats['trades_per_day'].median()
account_stats['freq_segment'] = account_stats['trades_per_day'].apply(lambda x: 'Frequent' if x > median_freq else 'Infrequent')

print(f"\nFrequency segments (median={median_freq:.1f} trades/day):")
fq_stats = account_stats.groupby('freq_segment').agg(
    count=('Account', 'count'),
    avg_pnl=('total_pnl', 'mean'),
    avg_win_rate=('win_rate', 'mean'),
    avg_volume=('total_volume', 'mean'),
).reset_index()
print(fq_stats.to_string(index=False))


Frequency segments (median=59.6 trades/day):
freq_segment  count       avg_pnl  avg_win_rate   avg_volume
    Frequent     16 383853.336943      0.835924 5.807757e+07
  Infrequent     16 257052.097389      0.864163 1.636610e+07


In [98]:
# Segment: Consistent winners vs losers
account_stats['pnl_segment'] = account_stats['total_pnl'].apply(
    lambda x: 'Winner' if x > 0 else ('Loser' if x < 0 else 'Breakeven')
)
print(f"\nPnL segments:")
pnl_seg = account_stats.groupby('pnl_segment').agg(
    count=('Account', 'count'),
    avg_pnl=('total_pnl', 'mean'),
    avg_win_rate=('win_rate', 'mean'),
    avg_volume=('total_volume', 'mean'),
    avg_trades=('total_trades', 'mean'),
).reset_index()
print(pnl_seg.to_string(index=False))


PnL segments:
pnl_segment  count       avg_pnl  avg_win_rate   avg_volume  avg_trades
      Loser      3 -89753.638695      0.706246 1.514237e+07 3075.000000
     Winner     29 362887.857427      0.864919 3.950592e+07 6965.275862


In [99]:
# How do winners vs losers behave differently on Fear vs Greed?
print("\n\n--- Winners vs Losers behavior by Sentiment ---")
winner_accounts = set(account_stats[account_stats['pnl_segment'] == 'Winner']['Account'])
loser_accounts = set(account_stats[account_stats['pnl_segment'] == 'Loser']['Account'])

for label, accs in [('Winners', winner_accounts), ('Losers', loser_accounts)]:
    subset = merged[merged['Account'].isin(accs)]
    if len(subset) == 0:
        continue
    print(f"\n  {label} ({len(accs)} accounts):")
    behav = subset.groupby('sentiment_group').agg(
        total_pnl=('Closed PnL', 'sum'),
        mean_pnl=('Closed PnL', 'mean'),
        total_trades=('Closed PnL', 'count'),
        avg_size=('Size USD', 'mean'),
        buy_pct=('Side', lambda x: (x == 'BUY').mean() * 100),
    ).reset_index()
    print(behav.to_string(index=False))



--- Winners vs Losers behavior by Sentiment ---

  Winners (29 accounts):
sentiment_group    total_pnl  mean_pnl  total_trades    avg_size   buy_pct
           Fear 3.942866e+06  50.45318         78149 7340.842697 50.385801
          Greed 5.315305e+06  60.58294         87736 4544.097410 46.507705
        Neutral 1.265577e+06  35.04978         36108 4799.801341 51.235183

  Losers (3 accounts):
sentiment_group      total_pnl    mean_pnl  total_trades    avg_size   buy_pct
           Fear  153400.136723   30.149398          5088 4742.440120 35.947327
          Greed -450004.265887 -175.851608          2559 5614.196542 66.901133
        Neutral   27343.213079   17.327765          1578 4392.164937 29.721166


## Key Insights

32 accounts identified. Out of these:

29 accounts are net winners (positive total PnL)
3 accounts are net losers (negative total PnL)

Losers actually profit during Fear ($153K) but lose massively during Greed (-$450K).

During Greed, losers buy aggressively (66.9% buy), chasing the trend, and get crushed. 

Winners maintain balanced positioning regardless of sentiment.

Higher volume ≠ better win rate. Low-volume traders have higher win rates (87.6% vs 82.4%), but high-volume traders earn more total PnL.




## Part C — Strategy Recommendations

Rule 1: "Avoid aggressive buying during Greed"  

Reason: The 3 losing accounts lost -$450K during Greed while buying 66.9% of the time. They chased overextended trends.

Rule 2: "Extreme Greed = best risk/reward for skilled traders"

Reason: Extreme Greed days show the highest win rate (89.2%) and mean PnL ($67.89/trade). But MAX LOSS is $117K vs $35K on Fear days.


## Bonus — Clustering

We cluster traders based on their trading behavior to identify different archetypes.

In [100]:
print(merged.columns)

Index(['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side',
       'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL',
       'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID',
       'Timestamp', 'date', 'value', 'classification', 'sentiment_group',
       'Net PnL'],
      dtype='object')


In [101]:
from sklearn.cluster import KMeans

# Prepare data using your actual column names
cluster_data = merged[['Size USD', 'Closed PnL']].dropna()

# Apply clustering
kmeans = KMeans(n_clusters=3, random_state=42)
merged['cluster'] = kmeans.fit_predict(cluster_data)

# Summary
cluster_summary = merged.groupby('cluster').agg(
    avg_size=('Size USD', 'mean'),
    avg_pnl=('Closed PnL', 'mean'),
    trades=('Closed PnL', 'count')
)

cluster_summary

,avg_size,avg_pnl,trades
cluster,,,
0,3.923579e+03,42.403419,210075
1,2.826782e+05,1132.724766,1122
2,2.366092e+06,3603.404334,21


### Clustering Results

- **Cluster 0 (Low-Risk, High-Volume Traders)**:
  - Average trade size: ~3,923 USD
  - Average PnL: ~42 USD
  - Number of trades: 210,075
  - These traders execute a large number of small trades with relatively low profit per trade.

- **Cluster 1 (Medium-Risk Traders)**:
  - Average trade size: ~282,678 USD
  - Average PnL: ~1,132 USD
  - Number of trades: 1,122
  - These traders take moderately large positions and achieve higher returns compared to Cluster 0.

- **Cluster 2 (High-Risk, Low-Frequency Traders)**:
  - Average trade size: ~2,366,092 USD
  - Average PnL: ~3,603 USD
  - Number of trades: 21
  - These traders take extremely large positions but trade very infrequently.

### Key Insights

1. **Higher Risk Leads to Higher Returns (but Lower Frequency)**
   - As trade size increases from Cluster 0 → 2, average PnL also increases significantly.
   - However, the number of trades drops sharply, indicating selective trading.

2. **Most Traders Are Low-Risk and High-Frequency**
   - Cluster 0 dominates with over 210,000 trades, showing that most traders prefer smaller, frequent trades.

3. **High-Risk Traders Are Rare but Impactful**
   - Cluster 2 has only 21 trades but very high average PnL, indicating high capital concentration.

### Conclusion

Trader behavior varies significantly across risk profiles:
- Low-risk traders rely on volume
- High-risk traders rely on large positions and selective trades

This highlights the importance of aligning trading strategy with risk appetite.